# 2. Candidate pairs, features, and validation
Exact normalized name, name-prefix, postal-code, and address-number blocks form the candidate set. RapidFuzz similarities and token overlaps are then scored by a balanced logistic model. The threshold is selected for macro F0.5, not accuracy.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'dataset').exists(): REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT / 'code/business_entity_resolution'))
from src.entity_resolution import prepare_frame, make_pairs, labelled_pairs, train_pair_model, best_f05_threshold, macro_f05

MAX_ROWS = 20_000  # set to None for a full run
train_dir = REPO_ROOT / 'dataset/train'
read = lambda name: pd.read_csv(train_dir / name, sep='\t', nrows=MAX_ROWS)
s1 = prepare_frame(read('train_source1.tsv'))
targets = pd.concat([read('train_source2.tsv'), read('train_source3.tsv')], ignore_index=True)
targets = prepare_frame(targets)
truth = read('train_ground_truth.tsv')

In [ ]:
pairs, candidate_map = make_pairs(s1, targets, max_candidates=250)
labelled, y = labelled_pairs(pairs, truth, negative_ratio=5)
train_ids, valid_ids = train_test_split(np.arange(len(labelled)), test_size=0.2, random_state=42, stratify=y)
model = train_pair_model(labelled.iloc[train_ids], y[train_ids])
validation_prob = model.predict_proba(labelled.iloc[valid_ids, 2:])[:, 1]
validation_labels = y[valid_ids]
validation_groups = labelled.iloc[valid_ids]['source1_entity_id'].to_numpy()
threshold, score = best_f05_threshold(validation_labels, validation_prob, validation_groups)
print({'candidate_pairs': len(pairs), 'positive_pairs': int(y.sum()), 'threshold': threshold, 'macro_f05': score})

In [ ]:
# Inspect the highest-scoring validation pairs and the threshold trade-off.
validation = labelled.iloc[valid_ids, :2].copy()
validation['label'] = validation_labels
validation['probability'] = validation_prob
display(validation.sort_values('probability', ascending=False).head(20))